# 18 — Multi-timeframe trendline trading simulation

A 1m chart that trades **fitted support and resistance lines from 1m, 5m, 15m and 1h at once**.

### The rule

A line gives way, and you take the break in the direction it broke:

| | |
|---|---|
| **long** | price breaks **up** through a **resistance** line |
| **short** | price breaks **down** through a **support** line |

"Breaks" is three conditions you can combine with `TL_COMBINE = "all"` or `"any"`; set any of them
to `0` to switch it off:

| | |
|---|---|
| `TL_CLOSE_N` | that many consecutive **closes** beyond the line — the standard break rule |
| `TL_FULL_M` | that many consecutive candles **entirely** beyond it, wick included — much stricter |
| `TL_TOUCH_K` | the line had been **touched** that many times before it broke. A quality gate on the line, not a trigger of its own: a line nobody respected is not a level |

**Stop** = the extreme of the last `STOP_N` bars before entry — the previous low for a long, the
previous high for a short. Lag-free, unlike a swing pivot, which cannot exist until `lr` bars after
the fact.

**Target** = the best exit this project has measured: **breakeven at 1R, target at 3R**
(`be_ladder@v1`). Worth +0.0095R over a plain 3R target, and the best simple exit change of the 29
tested. Trailing stops were the *worst* of the 29, capturing 37% of the favourable move against
87–93% for fixed targets. Change `BE_AT_RR` / `TP_RR` when you want to try something else.

### Every timeframe's line, on one 1m grid

Lines are fitted on each timeframe **in its own bars** and merged onto the 1m grid with the
project's anti-lookahead shift: a 1h line read off the bar stamped 12:00 is only knowable when that
bar **closes at 13:00**, so 1m bars see it from 13:00 and not one second earlier.

### Read this before you believe any number below

This is a **simulation of a rule you specified**, not evidence that the rule works. It has not been
through the screen every other idea here faced — no shuffled-within-day null, no metronome
benchmark, no per-window bootstrap, no trim-top-K. `docs/WHAT_WORKS.md` records that **trendline
breaks were already measured and carry no directional edge**: the hit rate cleared 0.50 in 36% of
102 cells, which is *below* chance, and 596 breaks over 20 months worked 50.2% of the time.

And on 1m the cost floor is severe. Cell 7 measures it directly on your own trades rather than
asserting it.


## 1. Config — edit here

In [ ]:
# the kernel starts in notebooks/, so put the repo root on the path (same as the other notebooks)
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd

SYMBOL, SOURCE = "BTCUSDT", "binance"

TF          = "1m"                       # the chart and the execution timeframe
LINE_TFS    = ("5m", "15m", "1h")        # ALSO trade lines fitted on these
VIEW_START, END = "2026-06-01", "2026-06-10"
WARMUP_START    = "2026-05-25"           # 1m data starts here so the 1m lines have pivots
TF_WARMUP_BARS  = 800                    # history for each extra timeframe, in ITS OWN bars

# --- the entry rule --------------------------------------------------------------------
TL_CLOSE_N  = 2          # consecutive CLOSES beyond the line   (0 = off)
TL_FULL_M   = 0          # consecutive candles ENTIRELY beyond  (0 = off)
TL_TOUCH_K  = 0          # the line must have had this many touches (0 = off)
TL_COMBINE  = "all"      # "all" = every enabled condition, "any" = one of them
TL_TOL_ATR  = 0.0        # how far past the line counts as past it, in ATR

TRADE_LONG, TRADE_SHORT = True, True     # a one-sided result is a directional bet, not an edge

# --- the exit --------------------------------------------------------------------------
STOP_N          = 20     # stop at the extreme of the last N bars before entry
STOP_BUFFER_PCT = 0.05   # pushed this far beyond it, so noise at the level does not stop you
BE_AT_RR        = 1.0    # move the stop to breakeven here (0 = never)
TP_RR           = 3.0    # final target

# --- costs and sizing ------------------------------------------------------------------
COSTS = dict(initial_cash=10_000, fee_bps=8, slippage_bps=1.5,
             sizing_mode="risk_pct_equity", sizing_value=1.0,
             exit_enabled=False,          # only the stop and the target close a trade
             max_open_trades=1)           # one at a time, so the equity curve is readable

# --- how the lines themselves are fitted -----------------------------------------------
TRENDLINE = dict(atr_mult=3.5, min_touches=3, lookback_pivots=16, line_tol_atr=0.6,
                 min_span_bars=40, max_age_bars=500, break_buffer_atr=0.0)

# --- chart -----------------------------------------------------------------------------
HEIGHT, MAX_PICKER, MAX_SHAPES_TF = 900, 24, 12
MAX_TRADE_PICKER = 40    # trades listed in the trade dropdown (most recent kept)
SHOW_SR_ZONES, ZONE_LOOKBACK = True, 2000

## 2. Data

The 1m frame carries a calendar warm-up so its own lines have pivots to fit; each extra timeframe
gets `TF_WARMUP_BARS` of **its own** bars, because a 1h line needs 1h bars and a fixed start date
would starve it.

In [ ]:
from quant.data import get_ohlcv, tf_frames

df = get_ohlcv(SYMBOL, TF, start=WARMUP_START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
shown = int((df.t >= pd.Timestamp(VIEW_START, tz="UTC")).sum())
print(f"{TF}: {len(df):,} bars   {df.t.min()}  ->  {df.t.max()}")
print(f"      {len(df)-shown:,} warm-up, {shown:,} tradable from {VIEW_START}\n")

FRAMES = tf_frames(SYMBOL, LINE_TFS, view_start=VIEW_START, end=END,
                   warmup_bars=TF_WARMUP_BARS, source=SOURCE) if LINE_TFS else {}
for k, v in FRAMES.items():
    print(f"   {k:>4}: {len(v):>7,} bars from {v.t.min():%Y-%m-%d}")

## 3. Fit every timeframe's lines onto the 1m grid

`mtf_trendlines` fits on each frame separately and merges the higher ones back with the
`shift_bars=1` anti-lookahead rule. You get `tl_res` / `tl_sup` for 1m and `5m__tl_res`,
`1h__tl_sup` and so on for the rest.

In [ ]:
from quant.indicators.trendlines import mtf_trendlines, trendline_level_cols

D = mtf_trendlines(df, FRAMES, **TRENDLINE)
PREFIXES = ("",) + tuple(f"{tf}__" for tf in FRAMES)
TF_NAMES = {"": TF, **{f"{tf}__": tf for tf in FRAMES}}

print("level columns now on the 1m grid:")
for side in ("long", "short"):
    print(f"   {side:>5} uses {trendline_level_cols(D, side)}")

## 4. Does each line exist, and how often would it fire?

Before running anything. A timeframe that is **live 0%** of the time has no line at all over this
window — that is the fitter refusing to draw one, not a fault, and no setting short of loosening
`TRENDLINE` will conjure it.

A line is retired the moment it breaks, so asking for two closes beyond it would ask the second bar
to compare against a level that no longer exists. The entry holds the last published value for
exactly the length of the confirmation window, which is why the 1m count below is not zero.

In [ ]:
from quant.scenarios.entries import _tl_entry

rows = []
for pref, name in TF_NAMES.items():
    r = {"tf": name}
    for side, w in (("long", "res"), ("short", "sup")):
        col = f"{pref}tl_{w}"
        r[f"{w}_live_%"] = round(100 * float(np.isfinite(D[col].to_numpy(float)).mean()), 1) \
            if col in D.columns else np.nan
        r[f"{side}_fires"] = int(_tl_entry(
            dict(tl_prefixes=(pref,), tl_close_n=TL_CLOSE_N, tl_full_m=TL_FULL_M,
                 tl_touch_k=TL_TOUCH_K, tl_combine=TL_COMBINE, tl_tol_atr=TL_TOL_ATR),
            side)(D).sum())
    rows.append(r)
print(pd.DataFrame(rows).set_index("tf").to_string())
print("\n`fires` counts BARS meeting the rule; consecutive bars collapse into one trade.")

## 5. Run it

Long and short are separate scenarios on the same frame, stops and targets, so neither side can
flatter the other. `trendline_trigger` then labels each trade with **which** timeframe's line fired
— the entry is an OR across four, so the engine only knows that something did.

In [ ]:
from quant.scenarios import run_pieces
from quant.scenarios.entries import trendline_trigger
from quant.strategies.conditions import raw

# The warm-up bars exist to give the lines pivots and the stop its lookback -- NOT to be traded.
# Without this the first entry lands a week before VIEW_START and every number below describes a
# window you did not ask about.
_vs = pd.Timestamp(VIEW_START, tz="UTC")
# compared in pandas, not numpy: a tz-aware column becomes an object array and will not
# compare against a datetime64 at all
IN_WINDOW = raw(lambda d: (pd.to_datetime(d["t"]) >= _vs).to_numpy(), "in_window")

PARAMS = dict(tl_prefixes=PREFIXES, tl_close_n=TL_CLOSE_N, tl_full_m=TL_FULL_M,
              tl_touch_k=TL_TOUCH_K, tl_combine=TL_COMBINE, tl_tol_atr=TL_TOL_ATR,
              stop_n=STOP_N, stop_buffer_pct=STOP_BUFFER_PCT,
              be_at_rr=BE_AT_RR, tp_rr=TP_RR)

sides = [s for s, on in (("long", TRADE_LONG), ("short", TRADE_SHORT)) if on]
RUNS = {s: run_pieces(D, f"mtf_trendline_{s}@v1", "high_of_n@v1", "be_ladder@v1",
                      params=PARAMS, cfg=COSTS, entry_filter=IN_WINDOW) for s in sides}

TRADES = pd.concat(
    [trendline_trigger(r.df, r.trades, PARAMS, prefix_names=TF_NAMES).assign(side_run=s)
     for s, r in RUNS.items()], ignore_index=True).sort_values("entry_time").reset_index(drop=True)

risk = (TRADES.entry_price - TRADES.stop_price).abs()
TRADES["R"] = TRADES.pnl / (risk * TRADES.qty)
TRADES["risk_%"] = 100 * risk / TRADES.entry_price
# ONE ID PER TRADE, carried by its entry, stop, target and exit on the chart. `cause` is the exact
# line it broke -- "5m·S7" is the 8th support line the 5m fitter published, and the chart's hover,
# its legend and its dropdown all call it that too.
TRADES["tid"] = [f"#{i+1:02d}" for i in range(len(TRADES))]
print(f"{len(TRADES)} trades   {TRADES.entry_time.min()}  ->  {TRADES.exit_time.max()}")
assert TRADES.entry_time.min() >= _vs, "an entry landed in the warm-up"

## 6. Results

`R` is the trade's P&L divided by what it risked, so a clean stop-out is about **−1R** and the
target is **+3R**. Anything worse than −1R is the cost of trading, not the stop failing — cell 7
separates the two.

In [ ]:
def headline(name, tr, st):
    return dict(run=name, trades=len(tr),
                win_rate=f"{st.get('win_rate_pct', 0):.1f}%",
                PF=round(st.get("profit_factor", np.nan), 2),
                total_R=round(tr.R.sum(), 2), avg_R=round(tr.R.mean(), 3),
                ret=f"{st.get('total_return_pct', 0):+.2f}%",
                maxDD=f"{st.get('max_drawdown_pct', 0):.2f}%",
                expectancy=round(st.get("expectancy_per_trade", np.nan), 2))

summary = [headline(s, TRADES[TRADES.side_run == s], RUNS[s].stats) for s in sides]
if len(sides) > 1:
    both = {k: v for k, v in headline("BOTH", TRADES, {}).items()}
    both.update(win_rate=f"{100*(TRADES.pnl > 0).mean():.1f}%",
                PF=round(TRADES.pnl[TRADES.pnl > 0].sum() / abs(TRADES.pnl[TRADES.pnl < 0].sum()), 2),
                ret=f"{100*TRADES.pnl.sum()/COSTS['initial_cash']:+.2f}%", maxDD="-", expectancy="-")
    summary.append(both)
print(pd.DataFrame(summary).set_index("run").to_string())

### Why each trade ended

In [ ]:
why = (TRADES.groupby(["side_run", "close_reason"])
       .agg(n=("R", "size"), total_R=("R", "sum"), avg_R=("R", "mean"),
            median_bars=("bars_held", "median")).round(2))
print(why.to_string())
print("""
  stop_loss         the stop was hit. After BE_AT_RR it may be a BREAKEVEN stop, so a small
                    loss here is the ladder working, not the trade going against you
  take_profit       reached TP_RR
  forced_close_end  still open when the data ran out — not a result, just the window ending
  signal / alt_exit unused here (exit_enabled=False, no secondary)""")

### Which timeframe's line was worth following

In [ ]:
by_tf = (TRADES.groupby(["trigger", "side_run"])
         .agg(n=("R", "size"), total_R=("R", "sum"), avg_R=("R", "mean"),
              win=("pnl", lambda z: (z > 0).mean())).round(3))
print(by_tf.to_string())
print("\nA `+` means both timeframes' lines broke on the same bar.")
print("Read these as descriptions of one window, not as a ranking: with a handful of trades per")
print("cell the differences are inside the noise, which is what the phase-12 screen exists to test.")

## 7. The cost floor, measured on these trades

The single most important number on a 1m chart. A tight stop means a large position for the same
1% risk, and fees scale with position size — so the cost of a round trip, expressed in R, can be a
large fraction of what you are risking. `p*` is the win rate you need just to break even.

In [ ]:
sl = TRADES[TRADES.close_reason == "stop_loss"]
fee_R = (TRADES.fees / (risk * TRADES.qty))
gross_R = ((TRADES.exit_price - TRADES.entry_price) * TRADES.qty
           * np.where(TRADES.side == "long", 1, -1)) / (risk * TRADES.qty)
c = float(fee_R.median())
print(f"median risk per trade      {TRADES['risk_%'].median():.3f}% of price")
print(f"median PRICE move on a stop {gross_R[TRADES.close_reason == 'stop_loss'].median():.2f}R"
      "   <- the stop itself, doing its job")
print(f"median COST per round trip  {c:.2f}R   <- fees + slippage, on top of every trade")
print(f"\nbreak-even win rate  p* = (1 + cost) / (1 + RR) = "
      f"(1 + {c:.2f}) / (1 + {TP_RR}) = {(1 + c) / (1 + TP_RR):.1%}")
print(f"actual win rate                                    {100*(TRADES.pnl > 0).mean():.1f}%")

## 8. The chart

Trendlines from every timeframe, S/R zones, and the trades. Four legend groups, because the
questions are different:

| | |
|---|---|
| ▲ ▼ | **entries**, hover for side, R, close reason and which timeframe triggered it |
| grey line | the **entry level**, running to the exit |
| red dotted | the **stop** |
| green dotted | the **target** |
| ✕ | the **exit**, green if it made money |

**Everything is named.** A line is `5m·S7` — the eighth support line the 5m fitter published — and a
trade is `#12`. Hover anything and the first thing it says is its id: a stop line tells you which
trade owns it, a trendline tells you which line it is, and a trade's entry tells you which line it
broke. The `cause` column in the table above is the same name.

**Five dropdowns.** The first three are notebook 17's — timeframe, pattern family, occurrence. The
last two are for trades: pick **longs / shorts / winners / losers**, then a trade. Selecting one
clears the chart to **that trade alone plus the line that caused it** — its entry, stop, target and
exit, and nothing else. Rows read `#07 S 1m·S59  -2.88R sl`: id, side, line, R, and how it closed.

A line that a trade actually broke is never hidden by `MAX_PICKER` or `MAX_SHAPES_TF`. The caps keep
the chart openable, but not at the price of hiding the answer to "what caused this trade".

In [ ]:
from quant.viz import pattern_figure

fig = pattern_figure(
    D, patterns=[], trendlines=True, trendline_kwargs=TRENDLINE,
    tf_frames=FRAMES, view_start=VIEW_START, emas=(),
    sr_zones=SHOW_SR_ZONES, zone_lookback_bars=ZONE_LOOKBACK,
    max_picker_instances=MAX_PICKER, max_shapes_per_tf=MAX_SHAPES_TF,
    max_trade_picker=MAX_TRADE_PICKER,
    trades=TRADES, trade_tp_rr=TP_RR, start_visible=("trades", "sr_zones"), chart_tf=TF,
    title=f"{SYMBOL} {TF}  {VIEW_START} to {END}  ·  "
          f"{'/'.join((TF,) + tuple(FRAMES))} trendline breaks  ·  {len(TRADES)} trades",
    height=HEIGHT)
print(f"{len(fig.data)} traces, {len(fig.to_json())/1e6:.2f} MB")
fig

### Open it in a browser — bigger, and it survives the notebook being closed

In [ ]:
from quant.viz import open_in_browser
open_in_browser(fig)

### Every trade

In [ ]:
cols = ["tid", "entry_time", "side", "cause", "entry_price", "stop_price", "exit_price",
        "risk_%", "R", "pnl", "bars_held", "close_reason"]
TRADES[cols].round(3).to_string()  # or TRADES.to_csv("trades.csv")

---

## 9. What to change, and what it does

| knob | effect |
|---|---|
| `TL_CLOSE_N` | more closes = fewer, later, better-confirmed entries. `1` is the raw break |
| `TL_FULL_M` | requires the whole candle past the line. Much stricter than a close; expect a large drop in count |
| `TL_TOUCH_K` | only trade lines that had been respected this many times |
| `TL_COMBINE` | `"all"` intersects the enabled conditions, `"any"` unions them |
| `TL_TOL_ATR` | a buffer past the line before it counts, in ATR. Filters marginal breaks |
| `STOP_N` | the lookback for the stop. Larger = wider stop = smaller position = **less cost in R** |
| `BE_AT_RR` / `TP_RR` | the exit ladder. `BE_AT_RR=0` disables the breakeven move |
| `LINE_TFS` | which timeframes contribute lines. `()` trades 1m lines only |
| `MAX_TRADE_PICKER` | how many trades the dropdown lists (most recent kept). Plotly menus clip rather than scroll, so this bounds the list, not the trading |

### If you want to make this fair rather than just plausible

The four methodology rules that bind everything else in this project, none of which this notebook
does yet:

1. **Difference it against a within-day-shuffled null** (`experiments/null_control.py`). If the same
   rule scores as well on shuffled bars, the structure was in the price, not in your rule.
2. **Make it beat the metronome** — `time_entry` enters every N bars with no signal at all, through
   the same stops, targets and costs. Most "real" entries lose to it.
3. **Count survivors against chance** — `n_configs × 0.5^n_windows`. Nine windows and thirty
   settings will hand you a "winner" by accident.
4. **Trim the top K trades, not just the top one.** Every fat-tailed result here collapsed once the
   best five of about forty-five were removed.
